# Presentation preparation — paper figures F & G

CellOT-style panels from **`evals_ood_data_space/evals.csv`** (Figure F; still uses `--setting ood` naming in CSV paths) and marginal KDEs (Figure G).

**Experiments looped:** **CD8 OOD** (group **a**) and **monocyte M2 OOD** under `results/hvg_{flavor}_{a|m2}_{ood}/`. Each line needs completed training + **`eval_dataspace`** (gene-space `imputed.h5ad`).

**Figure F** — **Data space**, **`ncells=80`** only: *R*² of feature means (`r2-means` squared) and **MMD**. Bars: **IMPACT_CellOT** vs **scGen**; panel **f**. Filenames use `figure_f_{experiment_stem}_{flavor}_dataspace_ncells80.*` (e.g. `figure_f_cd8_…`, `figure_f_m2_ood_…`).

**Figure G** — **Treated**, **IMPACT_CellOT**, **scGen**. **CD8** experiments use **`MARKER_PANEL`**; **M2** uses **`MYELOID_PANEL`**. Saves `figure_g_{stem}_{flavor}_dataspace.*`.

**Figure G caveat:** `imputed.h5ad` must be **gene space** (1000 HVG). If latent-sized, re-run evaluate with **`--embedding ae`**.

**R² scatter** — mean predicted vs actual (06-style): **scGen** and **IMPACT_CellOT** only, for each experiment × flavor; see section below.

**UMAP (eval context)** — Scanpy **`sc.pl.umap`** (see **06.3**): training atlas cells (`split==train`, all types), OOD mouse/human eval, predictions. Section below; references: **`06.3_scanpy_faceted_umap.ipynb`**, `06_cd8_holdout_evaluation.ipynb`, `14_hvg_flavor_notebook6_replica.ipynb`.


In [12]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad
from sklearn.model_selection import train_test_split

REPO = Path("/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT")
CELL_GPU = REPO / "cellot/cellot_gpu"
RESULTS = CELL_GPU / "results"
OUT_DIR = REPO / "speciesOT/baseline/analysis/presentation_figure_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FLAVOR_KEYS = ("seurat_v3", "pearson_residuals")

# Per-line config: toggle_ood group letter, train/eval mode, holdout ontology id(s),
# Figure F x-tick label, filename stem fragment, Figure G marker set (t vs myeloid).
EXPERIMENTS = {
    "cd8_ood": {
        "group": "a",
        "mode": "ood",
        "holdout": "CL:0000625",
        "xtick": "CD8",
        "stem": "cd8",
        "panel": "tcell",
        "actual_ylabel": "Actual CD8+ T cell",
        "atlas_umap_title": "CD8",
        "atlas_holdout_mouse_label": "CD8 holdout — mouse",
    },
    "m2_ood": {
        "group": "m2",
        "mode": "ood",
        "holdout": ("CL:0000875", "CL:0000576"),
        "xtick": "Non-classical +\ngeneric monocyte",
        "stem": "m2_ood",
        "panel": "myeloid",
        "actual_ylabel": "Actual monocyte (M2 holdout)",
        "atlas_umap_title": "M2 monocyte",
        "atlas_holdout_mouse_label": "Monocyte holdout — mouse",
    },
}

MARKER_PANEL = {
    "PTPRC (CD45)": "ENSG00000081237",
    "CD3E": "ENSG00000198851",
    "CD4": "ENSG00000010610",
    "CD8A": "ENSG00000153563",
    "CD5": "ENSG00000110448",
    "CD7": "ENSG00000173762",
    "CCR7": "ENSG00000126353",
    "NCAM1 (CD56)": "ENSG00000149294",
    "MS4A1 (CD20)": "ENSG00000156738",
    "CD14": "ENSG00000170458",
    "ITGAM (CD11b)": "ENSG00000169896",
}

MYELOID_PANEL = {
    "FCGR3A (CD16)": "ENSG00000143543",
    "MS4A7": "ENSG00000166926",
    "LILRB1": "ENSG00000104972",
    "LILRB2": "ENSG00000131042",
    "CX3CR1": "ENSG00000168329",
    "IFITM3": "ENSG00000185885",
    "IFITM2": "ENSG00000185201",
    "TNF": "ENSG00000232810",
    "HLA-DRA": "ENSG00000204287",
    "HLA-DRB1": "ENSG00000196126",
    "HLA-DPA1": "ENSG00000231389",
    "HLA-DPB1": "ENSG00000223865",
    "ITGAL (CD11a)": "ENSG00000169896",
    "ITGAX (CD11c)": "ENSG00000137776",
    "SELL (CD62L)": "ENSG00000188404",
    "TLR7": "ENSG00000196664",
    "TLR8": "ENSG00000196961",
    "NR4A1": "ENSG00000123358",
    "BCL2A1": "ENSG00000140379",
    "S100A10": "ENSG00000197747",
}


def marker_panel_for(experiment_key: str):
    lab = EXPERIMENTS[experiment_key]["panel"]
    return MARKER_PANEL if lab == "tcell" else MYELOID_PANEL


def result_dir_for(flavor_key: str, experiment_key: str) -> str:
    e = EXPERIMENTS[experiment_key]
    return f"hvg_{flavor_key}_{e['group']}_{e['mode']}"


def hvg_path_for(flavor_key: str, experiment_key: str) -> Path:
    g = EXPERIMENTS[experiment_key]["group"]
    return CELL_GPU / "datasets/speciesot-human-mouse-hvg" / f"hvg_{flavor_key}_{g}_v07.h5ad"

# Figure F — paper-like palette (IMPACT_CellOT red, scGen gray); optional future cAE ≈ "#9a9098"
COLORS_F = {"impact_cellot": "#f2555b", "scgen": "#c7bebe"}
LABELS_F = {"impact_cellot": "IMPACT_CellOT", "scgen": "scGen"}
FIG_F_DPI = 300

PALETTE_G = {
    "Treated": "#1f4e96",
    "IMPACT_CellOT": "#f05a5f",
    "scGen": "#c8c2c2",
}
ORDER_G = ["Treated", "IMPACT_CellOT", "scGen"]
DRAW_ORDER_G = ["scGen", "Treated", "IMPACT_CellOT"]  # KDE z-order: gray drawn first

NCELLS_EVAL = 80  # largest subsample in evals.csv (30, 50, 80)
FIG_G_NCOLS = 2  # compact 2-column grid (journal-style)
KDE_BW_ADJUST = 0.7
FIG_G_DPI = 300


## Helpers: toggle-OOD split (matches `cellot.data.cell.split_cell_data_toggle_ood`)


In [3]:
def add_transport(adatas, source="mouse", target="human", condition_col="condition"):
    m = {source: "source", target: "target"}
    out = adatas.copy()
    out.obs = out.obs.copy()
    out.obs["transport"] = out.obs[condition_col].map(m)
    return out[out.obs["transport"].notna()].copy()


def split_toggle_ood(adata, groupby, holdout, key, mode, random_state=0, test_size=0.2):
    split = pd.Series(index=adata.obs_names, dtype=object)
    for _, idx in adata.obs.groupby(groupby, observed=False).groups.items():
        tr, te = train_test_split(idx, random_state=random_state, test_size=test_size)
        split.loc[tr] = "train"
        split.loc[te] = "test"
    hv = [holdout] if isinstance(holdout, str) else list(holdout)
    ood_ix = adata.obs_names[adata.obs[key].isin(hv)]
    a, b = train_test_split(ood_ix, random_state=random_state, test_size=0.5)
    if mode == "ood":
        split.loc[a] = "ignore"
        split.loc[b] = "ood"
    else:
        split.loc[a] = "train"
        split.loc[b] = "ood"
    adata.obs["split"] = split.astype("category")
    return adata


def load_ood_holdout_frames(h5ad_path, holdout, mode):
    """Treated/received (human) vs source (mouse) for OOD split cells matching holdout ontology set."""
    raw = ad.read_h5ad(h5ad_path)
    d = add_transport(raw)
    d = split_toggle_ood(
        d,
        groupby="condition",
        holdout=holdout,
        key="cell_type_ontology_term_id",
        mode=mode,
        random_state=0,
        test_size=0.2,
    )
    genes = [str(g) for g in d.var_names]
    hv_ids = set([holdout] if isinstance(holdout, str) else list(holdout))
    m = (
        d.obs["cell_type_ontology_term_id"].astype(str).isin(hv_ids)
        & (d.obs["split"] == "ood")
        & (d.obs["transport"] == "source")
    )
    h = (
        d.obs["cell_type_ontology_term_id"].astype(str).isin(hv_ids)
        & (d.obs["split"] == "ood")
        & (d.obs["transport"] == "target")
    )
    Xm = d.X[m, :]
    Xh = d.X[h, :]
    if hasattr(Xm, "toarray"):
        Xm = Xm.toarray()
    else:
        Xm = np.asarray(Xm)
    if hasattr(Xh, "toarray"):
        Xh = Xh.toarray()
    else:
        Xh = np.asarray(Xh)
    source = pd.DataFrame(Xm, columns=genes)
    treated = pd.DataFrame(Xh, columns=genes)
    return treated, source, d.obs_names[m]


def aggregate_evals_csv(path, ncells=NCELLS_EVAL):
    df = pd.read_csv(path)
    sub = df[(df["nfeatures"] == "all") & (df["ncells"] == ncells)]
    out = {}
    for metric in ["r2-means", "mmd"]:
        v = sub.loc[sub["metric"] == metric, "value"].astype(float)
        if metric == "r2-means":
            v = v ** 2
        out[metric] = {
            "mean": float(v.mean()),
            "std": float(v.std(ddof=1)) if len(v) > 1 else 0.0,
        }
    return out


## Figure F — data space, per flavor × experiment (CD8 OOD / M2 OOD)


In [3]:
from matplotlib.ticker import MaxNLocator

ARROW_F = "#2f3f5f"

for exp_key, exp in EXPERIMENTS.items():
    xtick = exp["xtick"]
    stem_slug = exp["stem"]
    for flavor_key in FLAVOR_KEYS:
        result_dir = result_dir_for(flavor_key, exp_key)
        models = ["impact_cellot", "scgen"]
        x = np.arange(1)
        bw = 0.27
        tick_sz = 9
        ylab_sz = 10.5
        xlab_sz = 10.5
        leg_sz = 8.5
        xlim_bar = (-0.55, 0.55)
        panels = [
            ("r2-means", (0.0, 1.08)),
            ("mmd", None),
        ]
        by_metric = {mkey: [] for mkey, _ in panels}
        for model in models:
            p = RESULTS / result_dir / model / "evals_ood_data_space" / "evals.csv"
            if not p.exists():
                print("[skip F]", exp_key, flavor_key, "missing", p)
                break
            agg = aggregate_evals_csv(p)
            for mkey, _ in panels:
                by_metric[mkey].append(agg[mkey]["mean"])
        else:
            plt.rcParams.update({
                "font.family": "sans-serif",
                "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica", "Liberation Sans"],
                "axes.linewidth": 1.1,
                "xtick.major.width": 1.0,
                "ytick.major.width": 1.0,
            })
            fig, axes = plt.subplots(2, 1, figsize=(2.45, 4.65), sharex=True)
            ax_r2, ax_mmd = axes[0], axes[1]
            for ax, (mkey, ylim) in zip(axes, panels):
                for i, model in enumerate(models):
                    off = (i - 0.5) * bw
                    val = by_metric[mkey][i]
                    ax.bar(
                        x + off,
                        [val],
                        width=bw,
                        color=COLORS_F[model],
                        label=LABELS_F[model],
                        zorder=3,
                    )
                    xc = float(np.asarray(x).ravel()[0] + off)
                    ax.annotate(
                        f"{val:.3f}",
                        xy=(xc, val),
                        xytext=(0, 2),
                        textcoords="offset points",
                        ha="center",
                        va="bottom",
                        fontsize=4,
                        color="0.2",
                        clip_on=False,
                    )
                ax.set_xlim(*xlim_bar)
                ax.tick_params(axis="both", labelsize=tick_sz, direction="out", top=False, right=False)
                ax.spines["top"].set_visible(False)
                ax.spines["right"].set_visible(False)
                for side in ("left", "bottom"):
                    ax.spines[side].set_linewidth(1.1)
                if ylim is not None:
                    ax.set_ylim(*ylim)
            ax_mmd.yaxis.set_major_locator(MaxNLocator(nbins=4))
            ax_r2.set_ylabel(r"$R^2$ feature means", fontsize=ylab_sz)
            ax_mmd.set_ylabel("MMD", fontsize=ylab_sz)
            ax_r2.yaxis.set_label_coords(-0.30, 0.5)
            ax_mmd.yaxis.set_label_coords(-0.30, 0.5)
            ax_r2.text(
                -0.50,
                0.56,
                "↑",
                transform=ax_r2.transAxes,
                fontsize=11,
                color=ARROW_F,
                ha="center",
                va="center",
                clip_on=False,
            )
            ax_mmd.text(
                -0.50,
                0.50,
                "↓",
                transform=ax_mmd.transAxes,
                fontsize=11,
                color=ARROW_F,
                ha="center",
                va="center",
                clip_on=False,
            )
            ax_r2.set_xticks(x)
            ax_r2.tick_params(axis="x", which="both", bottom=False, labelbottom=False)
            ax_mmd.set_xticks(x)
            xtick_fs = tick_sz - 1 if "\n" in str(xtick) else tick_sz
            ax_mmd.set_xticklabels([xtick], fontsize=xtick_fs)
            ax_mmd.set_xlabel("OOD setting", fontsize=xlab_sz, labelpad=7)
            ax_r2.legend(
                frameon=False,
                loc="upper right",
                bbox_to_anchor=(1.8, 1.02),
                fontsize=leg_sz,
                handlelength=1.0,
                handletextpad=0.35,
                borderaxespad=0.0,
            )
            fig.subplots_adjust(left=0.33, right=0.86, top=0.96, bottom=0.20, hspace=0.24)
            fig.text(0.035, 0.975, "f", fontsize=18, fontweight="bold", ha="left", va="top")
            stem = OUT_DIR / f"figure_f_{stem_slug}_{flavor_key}_dataspace_ncells{NCELLS_EVAL}"
            save_kw = dict(bbox_inches="tight", pad_inches=0.03)
            fig.savefig(stem.with_suffix(".pdf"), **save_kw)
            fig.savefig(stem.with_suffix(".png"), **save_kw, dpi=FIG_F_DPI)
            plt.close(fig)
            print("saved", stem.with_suffix(".pdf"))


saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/presentation_figure_outputs/figure_f_cd8_seurat_v3_dataspace_ncells80.pdf


saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/presentation_figure_outputs/figure_f_cd8_pearson_residuals_dataspace_ncells80.pdf


saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/presentation_figure_outputs/figure_f_m2_ood_seurat_v3_dataspace_ncells80.pdf


saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/presentation_figure_outputs/figure_f_m2_ood_pearson_residuals_dataspace_ncells80.pdf


## Figure G — marginal densities (Treated vs IMPACT_CellOT vs scGen)


In [4]:
def marker_panel_hvg_split(columns, panel):
    """Order-preserving (panel dict order) list of (label, ENSG) present in HVG columns."""
    colset = set(str(c) for c in columns)
    present = [(lbl, e) for lbl, e in panel.items() if e in colset]
    missing = [(lbl, e) for lbl, e in panel.items() if e not in colset]
    return present, missing


def _imat_to_df(impact, genes_order):
    X = impact.X.toarray() if hasattr(impact.X, "toarray") else np.asarray(impact.X)
    return pd.DataFrame(X, index=impact.obs_names, columns=genes_order)


def plot_figure_g(flavor_key, experiment_key):
    from matplotlib.lines import Line2D

    exp = EXPERIMENTS[experiment_key]
    rd = result_dir_for(flavor_key, experiment_key)
    h5 = hvg_path_for(flavor_key, experiment_key)
    panel = marker_panel_for(experiment_key)

    if not h5.exists():
        print(f"[skip G] {experiment_key} {flavor_key}: missing HVG {h5}")
        return

    treated, _source, src_ix = load_ood_holdout_frames(h5, exp["holdout"], exp["mode"])
    genes_order = list(treated.columns)
    imp_path = RESULTS / rd / "impact_cellot" / "evals_ood_data_space" / "imputed.h5ad"
    sg_path = RESULTS / rd / "scgen" / "evals_ood_data_space" / "imputed.h5ad"
    if not imp_path.exists() or not sg_path.exists():
        print(f"[skip G] {experiment_key} {flavor_key}: missing imputed under {rd}")
        return

    impact = ad.read_h5ad(imp_path)
    scgen_ad = ad.read_h5ad(sg_path)

    Ximp = impact.X.toarray() if hasattr(impact.X, "toarray") else np.asarray(impact.X)
    if Ximp.shape[1] != len(genes_order):
        print(
            f"[skip G] {flavor_key}: IMPACT {imp_path.name} has shape {Ximp.shape} — expected "
            f"(*, {len(genes_order)}). Re-run evaluate with `--embedding ae`:\n"
            f"  python scripts/evaluate.py --outdir {RESULTS / rd / 'impact_cellot'} "
            f"--setting ood --where data_space --embedding ae"
        )
        return

    impact_df = _imat_to_df(impact, genes_order)
    assert impact_df.index.equals(src_ix), "IMPACT imputed rows must align with OOD mouse holdout source"

    Xsg = scgen_ad.X.toarray() if hasattr(scgen_ad.X, "toarray") else np.asarray(scgen_ad.X)
    if Xsg.shape[1] != len(genes_order):
        print(f"[skip G] {flavor_key}: scGen imputed has shape {Xsg.shape}, expected (*, {len(genes_order)})")
        return
    scgen_df = _imat_to_df(scgen_ad, genes_order)
    assert scgen_df.index.equals(src_ix), "scGen imputed rows must align with OOD mouse holdout source"

    present, missing = marker_panel_hvg_split(treated.columns, panel)
    n_panel = len(panel)
    print(f"\n[{experiment_key} / {flavor_key}] Figure G — {len(present)} / {n_panel} panel genes in HVG")
    for lbl, ensg in present:
        print(f"  IN HVG   {lbl:20s}  {ensg}")
    for lbl, ensg in missing:
        print(f"  missing  {lbl:20s}  {ensg}")
    if not present:
        print(f"[skip G] {experiment_key} {flavor_key}: no panel genes in HVG")
        return

    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica", "Liberation Sans"],
        "axes.linewidth": 1.0,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.major.size": 3.5,
        "ytick.major.size": 3.5,
    })

    ncols = FIG_G_NCOLS
    nrows = int(np.ceil(len(present) / ncols))
    fig_w = 4.6
    fig_h = 2.25 * nrows + 0.45
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), squeeze=False)

    def gene_series(seg):
        seg = str(seg)
        return {
            "Treated": treated[seg],
            "IMPACT_CellOT": impact_df[seg],
            "scGen": scgen_df[seg],
        }

    for idx, (label, ensg) in enumerate(present):
        r, c = divmod(idx, ncols)
        ax = axes[r][c]
        series = gene_series(ensg)
        combined = pd.concat([series[k] for k in ORDER_G], ignore_index=True)
        qlo, qhi = combined.quantile([0.005, 0.995])
        span = qhi - qlo
        pad = 0.06 * (span + 1e-12)

        z_for = {"scGen": 1, "Treated": 2, "IMPACT_CellOT": 3}
        for name in DRAW_ORDER_G:
            arr = series[name].to_numpy(dtype=float)
            arr = arr[np.isfinite(arr)]
            if arr.size == 0:
                continue
            spread = float(np.nanmax(arr) - np.nanmin(arr))
            tol = max(1e-12, 1e-9 * max(float(np.nanmax(np.abs(arr))), 1.0))
            if spread <= tol:
                ax.axvline(
                    float(np.nanmedian(arr)),
                    color=PALETTE_G[name],
                    lw=1.5,
                    zorder=z_for[name],
                )
            else:
                sns.kdeplot(
                    x=series[name],
                    ax=ax,
                    color=PALETTE_G[name],
                    bw_adjust=KDE_BW_ADJUST,
                    linewidth=1.5,
                    common_norm=False,
                    zorder=z_for[name],
                )
        ax.set_xlim(qlo - pad, qhi + pad)
        ax.set_title(label, fontsize=12, pad=4)
        ax.set_xlabel("")
        ax.set_ylabel("o.o.d." if c == 0 else "")
        ax.tick_params(axis="both", labelsize=9, direction="out", top=False, right=False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_linewidth(1.0)

    n_slots = nrows * ncols
    for j in range(len(present), n_slots):
        axes[j // ncols][j % ncols].axis("off")

    fig.subplots_adjust(left=0.14, right=0.98, top=0.90, bottom=0.10, wspace=0.42, hspace=0.55)
    fig.text(0.02, 0.98, "g", fontsize=18, fontweight="bold", va="top", ha="left")

    handles = [
        Line2D([0], [0], color=PALETTE_G[k], lw=2.2, label=k)
        for k in ORDER_G
    ]
    if len(present) < n_slots:
        je = len(present)
        er, ec = divmod(je, ncols)
        pos = axes[er][ec].get_position()
        leg_anchor = (pos.x0 + pos.width / 2, pos.y0 + pos.height / 2)
        leg_loc = "center"
    else:
        leg_anchor = (0.98, 0.04)
        leg_loc = "lower right"
    fig.legend(
        handles=handles,
        loc=leg_loc,
        bbox_to_anchor=leg_anchor,
        ncol=1,
        frameon=False,
        fontsize=9,
        handlelength=2.2,
    )

    stem_slug = exp["stem"]
    stem = OUT_DIR / f"figure_g_{stem_slug}_{flavor_key}_dataspace"
    save_kw = dict(bbox_inches="tight", pad_inches=0.04)
    fig.savefig(stem.with_suffix(".pdf"), **save_kw)
    fig.savefig(stem.with_suffix(".png"), **save_kw, dpi=FIG_G_DPI)
    plt.close(fig)
    print("saved", stem.with_suffix(".pdf"))


for ek in EXPERIMENTS:
    for fk in FLAVOR_KEYS:
        plot_figure_g(fk, ek)



[cd8_ood / seurat_v3] Figure G — 7 / 11 panel genes in HVG
  IN HVG   CD8A                  ENSG00000153563
  IN HVG   CD5                   ENSG00000110448
  IN HVG   CD7                   ENSG00000173762
  IN HVG   CCR7                  ENSG00000126353
  IN HVG   NCAM1 (CD56)          ENSG00000149294
  IN HVG   MS4A1 (CD20)          ENSG00000156738
  IN HVG   CD14                  ENSG00000170458
  missing  PTPRC (CD45)          ENSG00000081237
  missing  CD3E                  ENSG00000198851
  missing  CD4                   ENSG00000010610
  missing  ITGAM (CD11b)         ENSG00000169896


saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/presentation_figure_outputs/figure_g_cd8_seurat_v3_dataspace.pdf



[cd8_ood / pearson_residuals] Figure G — 5 / 11 panel genes in HVG
  IN HVG   PTPRC (CD45)          ENSG00000081237
  IN HVG   CD3E                  ENSG00000198851
  IN HVG   CD4                   ENSG00000010610
  IN HVG   CD8A                  ENSG00000153563
  IN HVG   CD7                   ENSG00000173762
  missing  CD5                   ENSG00000110448
  missing  CCR7                  ENSG00000126353
  missing  NCAM1 (CD56)          ENSG00000149294
  missing  MS4A1 (CD20)          ENSG00000156738
  missing  CD14                  ENSG00000170458
  missing  ITGAM (CD11b)         ENSG00000169896


saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/presentation_figure_outputs/figure_g_cd8_pearson_residuals_dataspace.pdf



[m2_ood / seurat_v3] Figure G — 5 / 20 panel genes in HVG
  IN HVG   CX3CR1                ENSG00000168329
  IN HVG   TNF                   ENSG00000232810
  IN HVG   HLA-DRA               ENSG00000204287
  IN HVG   TLR7                  ENSG00000196664
  IN HVG   NR4A1                 ENSG00000123358
  missing  FCGR3A (CD16)         ENSG00000143543
  missing  MS4A7                 ENSG00000166926
  missing  LILRB1                ENSG00000104972
  missing  LILRB2                ENSG00000131042
  missing  IFITM3                ENSG00000185885
  missing  IFITM2                ENSG00000185201
  missing  HLA-DRB1              ENSG00000196126
  missing  HLA-DPA1              ENSG00000231389
  missing  HLA-DPB1              ENSG00000223865
  missing  ITGAL (CD11a)         ENSG00000169896
  missing  ITGAX (CD11c)         ENSG00000137776
  missing  SELL (CD62L)          ENSG00000188404
  missing  TLR8                  ENSG00000196961
  missing  BCL2A1                ENSG00000140379
  missing 

saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/presentation_figure_outputs/figure_g_m2_ood_seurat_v3_dataspace.pdf



[m2_ood / pearson_residuals] Figure G — 3 / 20 panel genes in HVG
  IN HVG   SELL (CD62L)          ENSG00000188404
  IN HVG   NR4A1                 ENSG00000123358
  IN HVG   S100A10               ENSG00000197747
  missing  FCGR3A (CD16)         ENSG00000143543
  missing  MS4A7                 ENSG00000166926
  missing  LILRB1                ENSG00000104972
  missing  LILRB2                ENSG00000131042
  missing  CX3CR1                ENSG00000168329
  missing  IFITM3                ENSG00000185885
  missing  IFITM2                ENSG00000185201
  missing  TNF                   ENSG00000232810
  missing  HLA-DRA               ENSG00000204287
  missing  HLA-DRB1              ENSG00000196126
  missing  HLA-DPA1              ENSG00000231389
  missing  HLA-DPB1              ENSG00000223865
  missing  ITGAL (CD11a)         ENSG00000169896
  missing  ITGAX (CD11c)         ENSG00000137776
  missing  TLR7                  ENSG00000196664
  missing  TLR8                  ENSG00000196961
  

saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/presentation_figure_outputs/figure_g_m2_ood_pearson_residuals_dataspace.pdf


## R² scatter — mean predicted vs actual (notebook 06 style)

Per **experiment** (`cd8_ood`, `m2_ood`) × **HVG flavor**: two panels (**scGen**, **IMPACT_CellOT**). Gray = all HVG genes; **red** = top 100 by variance in human treated OOD; **blue** = biomarkers in **`MARKER_PANEL`** (CD8) or **`MYELOID_PANEL`** (M2) present in the HVG set, with **blue** text labels (readable names, no Ensembl). **$r^2_{\mathrm{all\ genes}}$** on each panel is read from **`evals_ood_data_space/evals.csv`** (`r2-means` squared; same **ncells** and aggregation as Figure F via **NCELLS_EVAL**), not recomputed from the scatter. The gold box remains **$r^2$** on the top 100 high-variance genes (computed on the plot). Saves `r2_scatter_{stem}_{flavor}.png` (dpi 500) and `.pdf` under `presentation_figure_outputs/`.


In [5]:
from numpy.polynomial.polynomial import polyfit

# Match folder names under RESULTS / hvg_{flavor}_{group}_ood /
R2_SCATTER_MODEL_EVALS_SUBDIR = {"scGen": "scgen", "IMPACT_CellOT": "impact_cellot"}


def plot_r2_mean_scatter(flavor_key: str, experiment_key: str):
    exp = EXPERIMENTS[experiment_key]
    stem_slug = exp["stem"]
    panel = marker_panel_for(experiment_key)
    ensg_to_display = {str(ensg): name for name, ensg in panel.items()}
    rd = result_dir_for(flavor_key, experiment_key)
    h5 = hvg_path_for(flavor_key, experiment_key)
    sg_path = RESULTS / rd / "scgen" / "evals_ood_data_space" / "imputed.h5ad"
    imp_path = RESULTS / rd / "impact_cellot" / "evals_ood_data_space" / "imputed.h5ad"

    if not h5.exists():
        print("[skip R2 scatter]", experiment_key, flavor_key, "missing", h5)
        return
    if not sg_path.exists() or not imp_path.exists():
        print("[skip R2 scatter]", experiment_key, flavor_key, "missing imputed under", rd)
        return

    treated, _source, src_ix = load_ood_holdout_frames(h5, exp["holdout"], exp["mode"])
    genes_order = list(treated.columns)
    scgen_ad = ad.read_h5ad(sg_path)
    impact_ad = ad.read_h5ad(imp_path)

    Xsg = scgen_ad.X.toarray() if hasattr(scgen_ad.X, "toarray") else np.asarray(scgen_ad.X)
    Xim = impact_ad.X.toarray() if hasattr(impact_ad.X, "toarray") else np.asarray(impact_ad.X)
    if Xsg.shape[1] != len(genes_order) or Xim.shape[1] != len(genes_order):
        print("[skip R2 scatter]", experiment_key, flavor_key, "imputed not in gene space")
        return
    scgen_ix = scgen_ad.obs_names
    imp_ix = impact_ad.obs_names
    if not scgen_ix.equals(src_ix) or not imp_ix.equals(src_ix):
        print("[skip R2 scatter]", experiment_key, flavor_key, "imputed obs do not match OOD source")
        return

    treated_df = treated.copy()
    actual_means = treated_df.mean(0)
    gene_var = treated_df.var(0)
    top_100_degs = gene_var.nlargest(100).index.tolist()

    predicted_means = {
        "scGen": pd.DataFrame(Xsg, columns=genes_order).mean(0),
        "IMPACT_CellOT": pd.DataFrame(Xim, columns=genes_order).mean(0),
    }
    pred0 = predicted_means["scGen"]
    shared0 = actual_means.index.intersection(pred0.index)
    is_deg0 = np.array([g in top_100_degs for g in shared0])
    if is_deg0.sum() < 3:
        print("[skip R2 scatter]", experiment_key, flavor_key, "<3 top-var genes in overlap")
        return
    r2_all_from_evals = {}
    for _disp, _subdir in R2_SCATTER_MODEL_EVALS_SUBDIR.items():
        eval_path = RESULTS / rd / _subdir / "evals_ood_data_space" / "evals.csv"
        if not eval_path.exists():
            print("[R2 scatter] missing evals.csv (using plot recomputation for R²_all):", eval_path)
            r2_all_from_evals[_disp] = None
        else:
            r2_all_from_evals[_disp] = aggregate_evals_csv(eval_path)["r2-means"]["mean"]
    n_models = len(predicted_means)
    fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5.5))
    if n_models == 1:
        axes = [axes]

    actual_yl = exp["actual_ylabel"]
    for ax, (model_name, pred) in zip(axes, predicted_means.items()):
        shared = actual_means.index.intersection(pred.index)
        x_all = pred[shared].values.astype(float)
        y_all = actual_means[shared].values.astype(float)
        is_deg = np.array([g in top_100_degs for g in shared])
        panel_mask = np.array([str(g) in ensg_to_display for g in shared])
        r_all = np.corrcoef(x_all, y_all)[0, 1]
        r2_all_val = r2_all_from_evals[model_name]
        if r2_all_val is None:
            r2_all_val = float(r_all**2)
        r_deg = np.corrcoef(x_all[is_deg], y_all[is_deg])[0, 1]
        r2_deg_val = r_deg ** 2

        ax.scatter(x_all, y_all, s=10, alpha=0.22, c="0.55", zorder=2, rasterized=True)
        ax.scatter(x_all[is_deg], y_all[is_deg], s=22, alpha=0.78, c="red", zorder=4)
        ax.scatter(
            x_all[panel_mask],
            y_all[panel_mask],
            s=36,
            alpha=0.9,
            c="tab:blue",
            edgecolors="navy",
            linewidths=0.35,
            zorder=6,
        )
        pad = 0.05
        lims = [min(x_all.min(), y_all.min()) - pad, max(x_all.max(), y_all.max()) + pad]
        b, m = polyfit(x_all[is_deg], y_all[is_deg], 1)
        xs = np.linspace(lims[0], lims[1], 100)
        ax.plot(xs, b + m * xs, color="red", alpha=0.4, linewidth=1.5)

        shared_list = list(shared)
        pos = {g: i for i, g in enumerate(shared_list)}
        for g in shared_list:
            lbl = ensg_to_display.get(str(g))
            if lbl is None:
                continue
            i = pos[g]
            ax.annotate(
                lbl,
                (x_all[i], y_all[i]),
                fontsize=9,
                alpha=0.95,
                xytext=(4, 4),
                textcoords="offset points",
                color="navy",
                fontweight="bold",
                fontstyle="italic",
            )

        ax.set_xlabel(f"Predicted ({model_name})", fontsize=15)
        ax.set_ylabel(actual_yl, fontsize=15)
        ax.set_title(model_name, fontsize=18, fontweight="bold")
        ax.tick_params(labelsize=12)
        ax.text(
            0.95,
            0.15,
            f"$r^2_{{all\\ genes}}$ = {r2_all_val:.2f}",
            transform=ax.transAxes,
            ha="right",
            fontsize=14,
        )
        ax.text(
            0.95,
            0.05,
            f"$r^2_{{top\\ 100\\ DEGs}}$ = {r2_deg_val:.2f}",
            transform=ax.transAxes,
            ha="right",
            fontsize=14,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="gold", alpha=0.5),
        )

    fig.suptitle(
        f"Mean gene expression: predicted vs actual ({experiment_key}, {flavor_key})",
        fontsize=14,
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    stem = OUT_DIR / f"r2_scatter_{stem_slug}_{flavor_key}"
    fig.savefig(stem.with_suffix(".png"), dpi=500, bbox_inches="tight")
    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight", pad_inches=0.04)
    plt.close(fig)
    print("saved", stem.with_suffix(".png"))


for ek in EXPERIMENTS:
    for fk in FLAVOR_KEYS:
        plot_r2_mean_scatter(fk, ek)


saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/presentation_figure_outputs/r2_scatter_cd8_seurat_v3.png


saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/presentation_figure_outputs/r2_scatter_cd8_pearson_residuals.png


saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/presentation_figure_outputs/r2_scatter_m2_ood_seurat_v3.png


saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/presentation_figure_outputs/r2_scatter_m2_ood_pearson_residuals.png


## UMAP — training atlas reference + scGen + IMPACT_CellOT (Pearson, CD8 / M2)

**Design:**

1. **Reference:** All **`split == "train"`** cells in the Pearson HVG object (`hvg_pearson_residuals_{a|m2}_v07.h5ad` per **EXPERIMENTS**). **Mouse vs human** from `transport` (source = mouse, target = human).
2. **UMAP:** PCA → neighbors → UMAP fit **only on that reference** (`min_dist=0.3`, `random_state=42`).
3. **Overlays:** **OOD mouse** (CD8 or M2 monocyte holdout), **scGen** and **IMPACT_CellOT** predicted human (`evals_ood_data_space/imputed.h5ad`), **actual human OOD** — projected with **kNN in PCA space** (same as **`06.3_scanpy_faceted_umap`**). Both imputed objects must be present; there is no scGen-only output for this figure.
4. **Colors:** Same overlay palette for all lines (`COLORS_F`-aligned choices in code); circular points, no edges; diagnostic-style legend (counts in labels).

Uses **`umap_adata_split`** from Step A when present; otherwise loads the H5 inline.

Output: `umap_atlas_ref_{stem}_pearson_scgen_impact.{png,pdf}` (e.g. **`cd8`**, **`m2_ood`**).


In [13]:
import scanpy as sc
from sklearn.neighbors import NearestNeighbors


def _x_dense(X):
    return X.toarray() if hasattr(X, "toarray") else np.asarray(X)


def project_onto_ref_umap(pred_X, ref_adata, n_neighbors=10):
    pcs = ref_adata.varm["PCs"]
    ref_pca = ref_adata.obsm["X_pca"]
    ref_umap = ref_adata.obsm["X_umap"]
    ref_mean = np.asarray(ref_adata.X.mean(axis=0)).ravel()
    pred_X = np.asarray(pred_X, dtype=np.float32)
    pred_pca = (pred_X - ref_mean) @ pcs
    k = min(n_neighbors, max(1, ref_pca.shape[0] - 1))
    nn = NearestNeighbors(n_neighbors=k, metric="euclidean")
    nn.fit(ref_pca)
    dists, idxs = nn.kneighbors(pred_pca)
    w = 1.0 / (dists + 1e-8)
    w = w / w.sum(axis=1, keepdims=True)
    return np.array([(w[i, :, None] * ref_umap[idxs[i]]).sum(axis=0) for i in range(len(pred_pca))])


def plot_atlas_umap_pearson_scgen_impact(experiment_key: str):
    flavor_key = "pearson_residuals"
    exp = EXPERIMENTS[experiment_key]
    rd = result_dir_for(flavor_key, experiment_key)
    h5 = hvg_path_for(flavor_key, experiment_key)
    sg_path = RESULTS / rd / "scgen" / "evals_ood_data_space" / "imputed.h5ad"
    imp_path = RESULTS / rd / "impact_cellot" / "evals_ood_data_space" / "imputed.h5ad"

    d = globals().get("umap_adata_split", {}).get((experiment_key, flavor_key))
    if d is None:
        if not h5.exists():
            print("[atlas UMAP]", experiment_key, "missing", h5)
            return
        raw = ad.read_h5ad(h5)
        d = add_transport(raw)
        d = split_toggle_ood(
            d,
            groupby="condition",
            holdout=exp["holdout"],
            key="cell_type_ontology_term_id",
            mode=exp["mode"],
            random_state=0,
            test_size=0.2,
        )
    if not sg_path.exists():
        print("[atlas UMAP]", experiment_key, "missing scGen", sg_path)
        return
    if not imp_path.exists():
        print("[atlas UMAP]", experiment_key, "missing IMPACT", imp_path)
        return

    hv_ids = set([exp["holdout"]] if isinstance(exp["holdout"], str) else list(exp["holdout"]))
    ct = d.obs["cell_type_ontology_term_id"].astype(str)

    ref = d[d.obs["split"] == "train"].copy()
    if ref.n_obs < 50:
        print("[atlas UMAP]", experiment_key, "too few training cells", ref.n_obs)
        return
    ref.X = _x_dense(ref.X).astype(np.float32)
    tr = ref.obs["transport"].astype(str)
    ref.obs["atlas_species"] = np.where(tr == "source", "Atlas — mouse (train)", "Atlas — human (train)")

    n_pcs = int(min(50, ref.n_vars - 1, max(2, ref.n_obs - 1)))
    sc.pp.pca(ref, n_comps=n_pcs)
    sc.pp.neighbors(ref, n_neighbors=min(15, max(2, ref.n_obs - 1)), n_pcs=n_pcs)
    sc.tl.umap(ref, min_dist=0.3, random_state=42)

    ood_m = (d.obs["split"] == "ood") & ct.isin(hv_ids) & (d.obs["transport"] == "source")
    ood_h = (d.obs["split"] == "ood") & ct.isin(hv_ids) & (d.obs["transport"] == "target")
    src_ix = d.obs_names[ood_m]
    X_m = _x_dense(d[ood_m].X).astype(np.float32)
    X_h = _x_dense(d[ood_h].X).astype(np.float32)

    sg = ad.read_h5ad(sg_path)
    imp = ad.read_h5ad(imp_path)
    X_sg = _x_dense(sg.X).astype(np.float32)
    X_im = _x_dense(imp.X).astype(np.float32)

    for name, adt in [("scGen", sg), ("IMPACT", imp)]:
        if not adt.obs_names.equals(src_ix):
            print(f"[atlas UMAP] {experiment_key} {name} imputed obs_names do not match OOD mouse source")
            return
    if X_sg.shape[1] != ref.n_vars or X_im.shape[1] != ref.n_vars:
        print("[atlas UMAP]", experiment_key, "gene dim mismatch")
        return

    u_m = project_onto_ref_umap(X_m, ref)
    u_h = project_onto_ref_umap(X_h, ref)
    u_sg = project_onto_ref_umap(X_sg, ref)
    u_im = project_onto_ref_umap(X_im, ref)

    _atlas_defaults = {
        "cd8_ood": ("CD8", "CD8 holdout — mouse"),
        "m2_ood": ("M2 monocyte", "Monocyte holdout — mouse"),
    }
    _t, _h = _atlas_defaults[experiment_key]
    atlas_title = exp.get("atlas_umap_title", _t)
    hold_lbl = exp.get("atlas_holdout_mouse_label", _h)

    fig, ax = plt.subplots(figsize=(9, 7))
    ur = ref.obsm["X_umap"]
    for lab, color, s, alpha in [
        ("Atlas — mouse (train)", "#aec7e8", 7, 0.32),
        ("Atlas — human (train)", "#98df8a", 7, 0.32),
    ]:
        m = ref.obs["atlas_species"].astype(str) == lab
        ax.scatter(
            ur[m, 0],
            ur[m, 1],
            s=s,
            c=color,
            alpha=alpha,
            label=f"{lab} (n={int(m.sum())})",
            edgecolors="none",
            zorder=1,
        )
    ax.scatter(
        u_m[:, 0],
        u_m[:, 1],
        s=24,
        c="#6baed6",
        alpha=0.9,
        edgecolors="none",
        label=f"{hold_lbl} (n={len(u_m)})",
        zorder=4,
    )
    ax.scatter(
        u_sg[:, 0],
        u_sg[:, 1],
        s=24,
        c="#800080",
        alpha=0.88,
        edgecolors="none",
        label=f"scGen predicted human (n={len(u_sg)})",
        zorder=5,
    )
    ax.scatter(
        u_im[:, 0],
        u_im[:, 1],
        s=24,
        c="#00FFFF",
        alpha=0.88,
        edgecolors="none",
        label=f"IMPACT_CellOT predicted human (n={len(u_im)})",
        zorder=6,
    )
    ax.scatter(
        u_h[:, 0],
        u_h[:, 1],
        s=24,
        c="#d62728",
        alpha=0.92,
        edgecolors="none",
        label=f"Actual human OOD (n={len(u_h)})",
        zorder=7,
    )
    ax.set_xlabel("UMAP1")
    ax.set_ylabel("UMAP2")
    ax.set_title(
        f"Training atlas UMAP (Pearson HVG, {atlas_title}) + model overlays\n"
        "(mouse/human atlas • holdout mouse • scGen & IMPACT predicted • actual human)",
        fontsize=10,
        fontweight="bold",
    )
    ax.legend(loc="best", fontsize=7, framealpha=0.94)
    plt.tight_layout()
    stem = OUT_DIR / f"umap_atlas_ref_{exp['stem']}_pearson_scgen_impact"
    fig.savefig(stem.with_suffix(".png"), dpi=400, bbox_inches="tight")
    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight", pad_inches=0.03)
    plt.close(fig)
    print("saved", stem.with_suffix(".png"))


def plot_atlas_umap_cd8_pearson_scgen_impact():
    plot_atlas_umap_pearson_scgen_impact("cd8_ood")


def plot_atlas_umap_m2_ood_pearson_scgen_impact():
    plot_atlas_umap_pearson_scgen_impact("m2_ood")


plot_atlas_umap_cd8_pearson_scgen_impact()
plot_atlas_umap_m2_ood_pearson_scgen_impact()


saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/presentation_figure_outputs/umap_atlas_ref_cd8_pearson_scgen_impact.png
saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/presentation_figure_outputs/umap_atlas_ref_m2_ood_pearson_scgen_impact.png


### Outputs

PDFs and PNGs land in `speciesOT/baseline/analysis/presentation_figure_outputs/` (Figure F, G, R² scatter, UMAP `umap_eval_*`, atlas-reference **`umap_atlas_ref_{cd8|m2_ood}_pearson_scgen_impact.*`**).
